# Treino do modelo IC (LLaMA 2 + LoRA) no Google Colab

Este notebook treina o modelo de metadados ambientais usando a GPU gratuita do Colab.
Na RTX 4050 local levava ~60h; aqui costuma levar **2 a 4 horas**.

**Antes de começar:**
1. Menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU** (T4 serve).
2. Coloque a pasta **completa** do projeto no seu Google Drive, em `MyDrive/projeto-ic-2`
   (se usar outro nome, ajuste a célula 3). A pasta precisa conter `src/`, `configs/`, `data/` (com os 663 JSONs
   em `data/raw_json/`) e `benchmarks/`; a célula 3 confere se está tudo lá.
3. Rode as células na ordem.

> Dica: se o Colab desconectar no meio do treino, é só reabrir, rodar as células 1 a 5 de novo
> e usar a **célula de RETOMAR** (os checkpoints ficam salvos no seu Drive).

## 1. Verificar a GPU

In [ ]:
!nvidia-smi

## 2. Conectar o Google Drive
Vai abrir um pop-up pedindo autorização da sua conta Google.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Entrar na pasta do projeto
Ajuste `PROJETO` se a pasta do projeto no Drive tiver outro nome. A célula também confere se a pasta está completa.

In [ ]:
import os
PROJETO = "/content/drive/MyDrive/projeto-ic-2"   # pasta do projeto no seu Drive
os.chdir(PROJETO)
print("Pasta atual:", os.getcwd())

# Confere se a pasta está completa (código + configs + glossário + benchmark)
essenciais = ["src/analytics/compute.py", "src/prompts/generator.py", "src/prompts/add_negatives.py",
              "src/rag/index.py", "src/training/train.py", "configs/dataset_v3.yaml",
              "configs/exp01_colab.yaml", "data/glossary/glossary.yaml", "benchmarks/gold_qa.jsonl"]
faltando = [p for p in essenciais if not os.path.exists(p)]
print("Arquivos do projeto:", "OK" if not faltando else f"FALTANDO {faltando} -> suba a pasta completa")
n_json = len([f for f in os.listdir("data/raw_json") if f.endswith(".json")]) if os.path.isdir("data/raw_json") else 0
print(f"JSONs em data/raw_json: {n_json}")
# Gerados nas fases anteriores (normal faltar se ainda não gerou/treinou)
for p in ["data/processed/dataset_v3.jsonl", "checkpoints/exp01/adapter_config.json"]:
    print(f"{p}: {'sim' if os.path.exists(p) else 'não'}")

## 4. Instalar as bibliotecas
O Colab já vem com PyTorch e CUDA. Instalamos o resto. Leva 1-2 minutos.

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets sentencepiece sacrebleu rouge-score pyyaml
print("\nBibliotecas instaladas.")
import torch
print("CUDA disponível:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma")

## 5. Fazer login no Hugging Face
O LLaMA 2 exige aceitar a licença e autenticar. Cole seu token quando pedir.
- Aceite a licença: https://huggingface.co/meta-llama/Llama-2-7b-chat-hf
- Pegue um token (tipo *Read*): https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 6. Gerar o dataset (com o glossário e os negativos)
Roda as 4 etapas: estatísticas → glossário → gerador de perguntas → adiciona negativos.
Se você já gerou o `dataset_v3.jsonl` no PC e o subiu para o Drive, pode **pular esta célula**.

In [ ]:
!python -m src.analytics.compute --in data/raw_json --out data/analytics/stats.json
!python -m src.glossary.build --json-dir data/raw_json --out data/glossary/glossary_full.json
!python -m src.prompts.generator --config configs/dataset_v3.yaml
!python -m src.prompts.add_negatives --dataset data/processed/dataset_v3.jsonl --json-dir data/raw_json --n 2500

# Confere o tamanho e a mistura final
import json
from collections import Counter
c = Counter()
for l in open("data/processed/dataset_v3.jsonl", encoding="utf-8"):
    c[json.loads(l)["meta"]["kind"].split("_")[0]] += 1
print("Total:", sum(c.values()), "| mistura:", dict(c))

## 6b. (Opcional) Mais metadados da ARM — auditoria e serviços oficiais
A **auditoria** confere o corpus (registros válidos, duplicatas, cobertura dos campos, registros do Brasil) e não usa a internet.

O **catálogo** consulta serviços oficiais da ARM (os mesmos que a biblioteca ACT usa): variáveis com **unidade** (DOD), **relatórios de qualidade** (DQR) e **sítios com coordenadas**. O Colab tem acesso à internet, então é um bom lugar para rodar. Comece pequeno, confira, e depois rode completo — o cache fica no Drive (`data/arm_extra/cache`) e uma nova execução só busca o que falta.

O índice RAG (célula 8b) usa o catálogo automaticamente. Para o **novo fine-tuning**, refaça a célula 6 trocando a linha do glossário por `!python -m src.glossary.build --json-dir data/raw_json --out data/glossary/glossary_full.json --extra auto`.

In [ ]:
# 1) auditoria do corpus (sem internet)
!python -m src.collect.audit

# 2) catálogo dos serviços oficiais da ARM: teste pequeno primeiro
!python -m src.collect.arm_extra --max-classes 20
# depois, tudo (pode levar alguns minutos na 1ª vez):
# !python -m src.collect.arm_extra

## 7. Treinar
Os checkpoints são salvos no seu Drive a cada 500 passos, então nada se perde se cair.
Vai aparecer o progresso com ETA. Numa T4, espere ~2-4h para as 3 épocas.

In [ ]:
!python -m src.training.train --config configs/exp01_colab.yaml

## 7b. RETOMAR (só se o Colab caiu no meio do treino)
Rode as células 1 a 5 de novo, depois esta. Ela continua do último checkpoint salvo no Drive.

In [ ]:
!python -m src.training.train --config configs/exp01_colab.yaml --resume

## 8. Avaliar
Roda o benchmark e mostra as métricas (F1, BLEU, ROUGE-L, acurácia de recusa).

In [ ]:
!python -m src.eval.run_eval --config configs/exp01_colab.yaml --benchmark benchmarks/gold_qa.jsonl

## 8b. RAG — arquitetura híbrida (fase 3)
Em vez de exigir que o modelo **memorize** DOIs, datas e coordenadas, a busca encontra a ficha certa nos JSON-LD e a coloca no prompt para o modelo **ler**.

1. A primeira célula monta o índice e mede se a busca acha a ficha certa (**Recall@k**). Não precisa de GPU e leva segundos.
2. A segunda roda o benchmark com o RAG ligado — compare com o resultado da célula 8 (só fine-tuning).

In [ ]:
!python -m src.rag.build_index --json-dir data/raw_json --glossary data/glossary/glossary_full.json --out data/rag/index.json
!python -m src.rag.eval_retrieval --index data/rag/index.json --dataset data/processed/dataset_v3.jsonl

In [ ]:
# fine-tuning + RAG (a arquitetura híbrida)
!python -m src.eval.run_eval --config configs/exp01_colab.yaml --benchmark benchmarks/gold_qa.jsonl --rag
# (opcional) modelo base + RAG, sem o LoRA — para separar o efeito de cada parte
# !python -m src.eval.run_eval --config configs/exp01_colab.yaml --benchmark benchmarks/gold_qa.jsonl --baseline --rag

### 8c. Recalcular avaliações antigas com as métricas corrigidas
A fase 3 corrigiu a detecção de recusas (duas frases de recusa do treino e a resposta "fora do escopo" não eram reconhecidas). Isto re-pontua os resultados já salvos, **sem rodar o modelo de novo**.

In [ ]:
!python -m src.eval.rescore "benchmarks/results/eval_*.json"

## 9. Conversar com o modelo (teste rápido)
Faz algumas perguntas direto pelo notebook para você ver a qualidade.

In [ ]:
import torch, yaml
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from src.common.prompts import build_chat_prompt

cfg = yaml.safe_load(open("configs/exp01_colab.yaml"))
m = cfg["model"]
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
base = AutoModelForCausalLM.from_pretrained(m["base"], quantization_config=bnb, device_map="auto")
model = PeftModel.from_pretrained(base, m["output_dir"]); model.eval()
tok = AutoTokenizer.from_pretrained(m["output_dir"])
if tok.pad_token_id is None: tok.pad_token = tok.eos_token
sistema = cfg["system_prompt"]

def perguntar(q):
    prompt = build_chat_prompt(sistema, [], new_user_message=q)
    ins = tok(prompt, return_tensors="pt", truncation=True, max_length=1536).to(model.device)
    with torch.inference_mode():
        out = model.generate(**ins, max_new_tokens=256, do_sample=False,
                             repetition_penalty=1.05, pad_token_id=tok.eos_token_id)
    print("P:", q)
    print("R:", tok.decode(out[0, ins["input_ids"].shape[-1]:], skip_special_tokens=True).strip())
    print()

for q in ["O que é mean_doppler_velocity?",
          "Para que serve reflectivity?",
          "Qual a cotação do dólar hoje?",
          "Preciso de dados de umidade em Southern Great Plains. O que uso?"]:
    perguntar(q)

## 9b. Conversar com o modelo usando RAG
Rode a célula 9 antes (ela carrega o modelo). Abaixo de cada resposta aparecem as fichas consultadas.

In [ ]:
from src.rag.pipeline import RagPipeline

rag = RagPipeline.from_config(cfg)
tok.truncation_side = "left"   # se o prompt passar do limite, corta o início, nunca a pergunta

def perguntar_rag(q):
    res = rag.build_prompt(sistema, q, count_tokens=lambda s: len(tok(s)["input_ids"]))
    ins = tok(res.prompt, return_tensors="pt", truncation=True,
              max_length=rag.settings.max_input_tokens).to(model.device)
    with torch.inference_mode():
        out = model.generate(**ins, max_new_tokens=256, do_sample=False,
                             repetition_penalty=1.05, pad_token_id=tok.eos_token_id)
    print("P:", q)
    print("R:", tok.decode(out[0, ins["input_ids"].shape[-1]:], skip_special_tokens=True).strip())
    print("Fontes:", "; ".join(f"[{s['n']}] {s['title']}" for s in res.sources) or "nenhuma")
    print()

for q in ["Qual o período de coleta do dataset 1twrmr?",
          "Qual o DOI do dataset 1twrmr?",
          "O que significa a sigla KAZR?",
          "Qual a cotação do dólar hoje?"]:
    perguntar_rag(q)

## 10. (Opcional) Baixar o modelo treinado para o PC
O adapter LoRA já está no seu Drive (`checkpoints/exp01`). Se quiser um .zip para baixar:

In [ ]:
import os, shutil
shutil.make_archive("/content/adapter_exp01", "zip",
                    os.path.abspath("checkpoints/exp01"))   # na pasta do projeto
from google.colab import files
files.download("/content/adapter_exp01.zip")